# 03 - Downstream QC, Normalization, and Dimensionality Reduction

## Imports and configuration

In [ ]:
import os
import warnings
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from config import (
    ANNDATA_PROCESSED,
    ANNDATA_RAW,
    ANNDATA_V1,
    GENE_NAME_FIXES,
    LEIDEN_RESOLUTION,
    MIN_GENES_PER_CELL,
    MIN_UMI_PER_GENE,
    MT_FRAC_CUTOFF,
    N_HVG,
    N_NEIGHBORS,
    N_PCS,
    N_REGRESS_JOBS,
    NEIGHBOR_METRIC,
    PARENT_DIR,
    PROJECT_DIR,
    REGRESS_OUT_VARS,
)

warnings.filterwarnings("ignore")
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False)

os.chdir(PROJECT_DIR)

## 1. Load data

In [ ]:
##There are 90 low quality cells which form a distict cluster, remove them
cellsToDiscard = list(pd.read_csv("./TextFiles/cellsToDiscard.csv").iloc[:,1])

In [ ]:
geneModules = pd.read_csv(
    str(PARENT_DIR / "TextFiles" / "ME_GeneModules_leiden_8_Modules.csv"),
    index_col=0,
)
geneModules = geneModules["GeneName"]

In [ ]:
geneModules = geneModules.replace(GENE_NAME_FIXES)

In [ ]:
adata_v2 = sc.read_h5ad(ANNDATA_RAW)
adata_v2.obs["batch"] = "v2"

In [ ]:
adata_v2

In [ ]:
adata_v1 = sc.read_h5ad(ANNDATA_V1)
adata_v1.obs["batch"] = "v1"

In [ ]:
adata_v1

In [ ]:
adata = sc.AnnData.concatenate(adata_v1, adata_v2)

In [ ]:
adata

In [ ]:
guideCounts = pd.DataFrame(adata.obs["Guide"].value_counts())
guideCounts

In [ ]:
fig, ax = plt.subplots(figsize=(40, 20))
ax.bar(range(len(guideCounts)), guideCounts["Guide"].values, color="steelblue")
ax.set_xticks(range(len(guideCounts)))
ax.set_xticklabels(guideCounts.index, rotation=90, fontsize=25)
ax.tick_params(axis="y", labelsize=35)
ax.set_yticks(np.arange(0, guideCounts["Guide"].max() + 100, 100))
ax.set_xlabel("Guide names", fontsize=45)
ax.set_ylabel("Number of cells", fontsize=45)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
adata.obs["Guide"].value_counts().to_csv("ComboCellCounts_beforeFiltering.csv")

## 2. Inspect controls

In [ ]:
adataControl = adata[adata.obs["NumControlGuides"] > 0,]
adataControl

## 3. Cell QC filtering

In [ ]:
adata = adata[[x for x in adata.obs.index if x not in cellsToDiscard],:]

In [ ]:
adata

In [ ]:
adata.obs["SampleID"] = [str(x).split("-")[2] for x in adata.obs.index]

In [ ]:
adata.obs['n_umis'] = adata.X.sum(1)
adata.obs['n_genes'] = (adata.X != 0).sum(1).A1
adata.obs['log10_n_umis'] = np.log10(adata.X.sum(1))
adata.obs['log10_n_genes'] = np.log10((adata.X != 0).sum(1).A1)

mt_gene_mask = adata.var_names.str.startswith('mt-')

adata.obs['mt_frac'] = adata.X[:, mt_gene_mask].sum(1).A1 / adata.obs['n_umis']

In [ ]:
adata.obs['mt_frac'].describe()

In [ ]:
adata.obs['n_genes'].describe()

In [ ]:
adata.obs['n_umis'].describe()

In [ ]:
adata = adata[adata.obs['mt_frac'] < MT_FRAC_CUTOFF,]
adata = adata[adata.obs['n_genes'] > MIN_GENES_PER_CELL,]

In [ ]:
adata

## 4. Gene filtering

In [ ]:
k = pd.DataFrame(adata.X.A.sum(axis=0))

In [ ]:
MIN_UMI_PER_GENE

In [ ]:
selGenes = adata.var_names[adata.X.A.sum(axis=0) > MIN_UMI_PER_GENE]
selGenes = list(set(list(selGenes) + list(geneModules)))
selGenes = [x for x in selGenes if x in adata.var_names]
len(selGenes)

In [ ]:
adata = adata[:,selGenes]
adata

## 5. Normalization and preprocessing

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=adata.obs['n_umis'].median())
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HVG,
    layer="counts",
    flavor="seurat_v3",
)

In [ ]:
gene_list_url = 'https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt'

cell_cycle_genes = [str(x.strip(), 'utf-8').capitalize() for x in urlopen(gene_list_url)]

s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]

sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
sc.pp.combat(adata, key='batch')

In [ ]:
sc.pp.regress_out(adata, REGRESS_OUT_VARS, n_jobs=N_REGRESS_JOBS)

## 6. Dimensionality reduction and clustering

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=N_PCS)
sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, metric=NEIGHBOR_METRIC)
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION)

In [ ]:
sc.pl.umap(adata, color='leiden', vmax=0.5, cmap='Reds', legend_loc='on data')

In [ ]:
sc.pl.umap(adata, color='batch', vmax=0.5, cmap='Reds')

In [ ]:
sc.pl.umap(adata, color='phase', vmax=0.5, cmap='Reds')

In [ ]:
sc.pl.umap(adata, color='n_genes', cmap='Reds')

In [ ]:
sc.pl.umap(adata, color='n_genes', cmap='coolwarm')

In [ ]:
sc.pl.umap(adata, color='S_score', vmax=0.5, cmap='coolwarm')

In [ ]:
sc.pl.umap(adata, color='G2M_score', vmax=0.5, cmap='coolwarm')

In [ ]:
sc.pl.umap(adata, color='phase', vmax=0.5, cmap='Reds')

In [ ]:
sc.pl.umap(adata, color='mt_frac', vmax=0.2, cmap='Reds')

## 7. Cluster diagnostics

In [ ]:
pd.crosstab(adata.obs["leiden"], adata.obs["Guide"])

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (8, 3)}):
    sc.pl.violin(adata, ['mt_frac'],
                stripplot=False, inner='box', groupby='leiden')

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (8, 3)}):
    sc.pl.violin(adata, ['n_genes'],
                stripplot=False, inner='box', groupby='leiden')

## 8. Cell-type scoring

In [ ]:
cGenes = pd.read_csv(str(PARENT_DIR / 'TextFiles' / 'PositiveControls' / 'DC_cellstate_genes.csv'))

In [ ]:
cGenes

In [ ]:
for i in [col for col in cGenes.columns if 'genes' in col]:
    myGeneList = [x for x in cGenes.loc[:,i] if x != ' ']
    myGeneList = [x.replace(".","-") for x in myGeneList]
    if sum(pd.Series(myGeneList).isin(adata.var_names)) > 1:
        sc.tl.score_genes(adata=adata, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adata, color=i, size=10, color_map="coolwarm")
        sc.pl.violin(adata, i, groupby='leiden')

## 9. Embedding density

In [ ]:
adata.obs.columns[0:16]

In [ ]:
for elem in adata.obs.columns[0:16]:
    adata.obs["SelGuide"] = "Others"
    adata.obs.loc[adata.obs["Guide"] == elem, "SelGuide"] = elem
    sc.tl.embedding_density(adata, groupby="SelGuide")
    sc.pl.embedding_density(adata,
                        key="umap_density_SelGuide",
                        color_map="nipy_spectral")

## 10. Save processed data

In [ ]:
anndataProcessedFileName = ANNDATA_PROCESSED

In [ ]:
adata.write(anndataProcessedFileName)